# 08 · Post-experiment error analysis

Why the contradiction-aware pipeline (A3) did not outperform A2. Read-only over the frozen run: no prompt is changed, no model is called, nothing in the run directory or the reference is modified. Every number below is recomputed from the run's predictions and traces.

**Observed vs interpreted.** Tables and counts are observations. The *mechanism* column of the discordant-case review (section 6) is an analyst judgement made from the traces by an AI assistant and is **not researcher-verified**; the research findings report (`docs/error_analysis_report.md`) separates the two explicitly.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
RUN_ID = None
EXPERIMENT_ID = "claimverif-main-v1"
A, B = "A2_rag_numeric", "A3_contradiction_aware"   # main comparison (B = second look on top of A)

In [ ]:
import os
from engine import error_analysis as ea, plots
from engine.freeze import build_freeze, diff_freeze, freeze_path
from engine.metrics import classification_metrics
from engine.notebook import banner, pick_run, describe_run, load_gold_labels, reference_banner, trace_index, save_table, save_figure
from engine.reference import alternative_variants, load_alternatives
from engine.runner import RUNS_DIR
run = pick_run(EXPERIMENT_ID, RUN_ID)
assert run is not None, "no run found -- execute notebook 03 first"
describe_run(run)
reference_banner()
RID = run["manifest"]["run_id"]
preds, traces, gold = run["predictions"], trace_index(run), load_gold_labels()
kind_of = {p["approach"]: p["approach_kind"] for p in preds}
LABELLED = bool({p["claim_id"] for p in preds} & set(gold)) and {A, B} <= set(kind_of)
if not LABELLED:
    banner("This run has no reference-labelled claims for both compared approaches (the reference covers the gold "
           "split only); the label-based sections are skipped.")
plots.style()

## 1. Freeze check: the original results are unchanged

`scripts/freeze_experiment.py` stored checksums of the run directories, configuration, prompts, model settings, datasets, reference, adjudication files, cached model responses and result files, plus the scores recomputed from the frozen predictions. Here everything is recomputed and compared.

In [ ]:
FREEZE = freeze_path(RID)
frozen = json.load(open(FREEZE)) if os.path.exists(FREEZE) else None
SOURCE_RUNS = list(frozen["runs"])[1:] if frozen else []
if frozen is None:
    banner(f"No freeze record for {RID}: run scripts/freeze_experiment.py --run {RID} before reporting.", "danger")
else:
    now = build_freeze(RID, SOURCE_RUNS)
    d = diff_freeze(frozen, now)
    print("frozen at", frozen["frozen_at"], "| git", frozen["git"]["commit"][:10], "dirty" if frozen["git"]["dirty"] else "clean")
    print("changed:", d["changed"] or "none", "| removed:", d["removed"] or "none", "| added since freeze:", len(d["added"]))
    assert not d["changed"] and not d["removed"], "frozen inputs or results changed -- stop and investigate"
    print("model:", frozen["model"]["model"], "temperature", frozen["model"]["temperature"],
          "| prompts", frozen["versions"]["prompt_version"], "| prompt drift:", frozen["prompts"]["drift"] or "none",
          "| cached responses:", frozen["llm_cache"]["entries"], "missing", len(frozen["llm_cache"]["missing"]))
    display(pd.DataFrame(frozen["scores"]["accuracy"]).round(4))

## 2. Sensitivity scorings: right adjudication file, right provenance

Each recorded variant must name its source workbook, and re-reading that workbook must give the same decisions. Strict scoring uses no adjudication file.

In [ ]:
adj_dir = os.path.join("datasets", "adjudication")
prov = []
for v in alternative_variants(RID):
    path = os.path.join(adj_dir, f"insufficient_alternatives__{RID}__{v}.jsonl")
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    for book in sorted({r.get("source_workbook") for r in rows}):
        chk = ea.adjudication_matches_workbook(path, os.path.join(adj_dir, book))
        prov.append({"variant": v, "source_workbook": book, "adjudicator": ",".join(sorted({r.get("adjudicator") for r in rows})),
                     **{k: chk[k] for k in ("cases_recorded", "acceptable_recorded", "acceptable_in_workbook", "consistent")}})
prov = pd.DataFrame(prov)
display(prov if not prov.empty else "no adjudication variants recorded for this run")
if not prov.empty:
    assert prov.consistent.all(), "a recorded variant disagrees with its source workbook"
DEPRECATED = os.path.join("results", "DEPRECATED.json")
dep = [d for d in (json.load(open(DEPRECATED))["artifacts"] if os.path.exists(DEPRECATED) else []) if d["run_id"] == RID]
for d in dep:
    banner(f"{d['path']} is {d['status']}: {d['problem']} Use {', '.join(os.path.basename(x) for x in d['superseded_by'])}. "
           f"{d['action']}", "danger")
if dep:
    display(pd.DataFrame(dep)[["path", "status", "superseded_by"]])
current = os.path.join("results", "metrics", f"{RID}__accuracy_by_scoring.csv")
if os.path.exists(current) and frozen:
    t = pd.read_csv(current).set_index("scoring")
    s = t[[c for c in t.columns if not c.endswith("accepted")]].round(6).T.to_dict()
    assert s == frozen["scores"]["accuracy"], "accuracy_by_scoring.csv disagrees with the scores recomputed at freeze time"
    print("accuracy_by_scoring.csv matches the frozen strict / adjudicated / lenient scores")

## 3. A2 vs A3, claim by claim (strict scoring)

A3 starts from A2's final verdict (same initial request, same numeric verification), so every A2/A3 disagreement is a verdict the A3 second look changed.

In [ ]:
if LABELLED:
    pair = pd.DataFrame(ea.paired_outcomes(preds, gold, A, B))
    same_start = sum(1 for p in preds if p["approach"] == B and p.get("initial_label") ==
                     next((q["predicted_label"] for q in preds if q["approach"] == A and q["claim_id"] == p["claim_id"]), None))
    print(f"A3 initial verdict == A2 final verdict on {same_start}/{len(pair)} claims")
    q = pair.quadrant.value_counts().reindex(ea.QUADRANTS, fill_value=0)
    display(pd.crosstab(pair.correct_a.map({True: f"{A[:2]} correct", False: f"{A[:2]} wrong"}),
                        pair.correct_b.map({True: f"{B[:2]} correct", False: f"{B[:2]} wrong"}), margins=True))
    by_cat = pair.pivot_table(index="category", columns="quadrant", values="claim_id", aggfunc="count", fill_value=0) \
        .reindex(columns=list(ea.QUADRANTS), fill_value=0).reindex(list(plots.CATEGORIES))
    display(by_cat)
    save_table(pair, "ea_a2_a3_paired_claims", run)
    rows = [{"group": "All claims", **q.to_dict()}] + [{"group": c.capitalize(), **by_cat.loc[c].to_dict()} for c in by_cat.index]
    fig = plots.paired_quadrants(rows, (A, kind_of[A]), (B, kind_of[B]), "A2 vs A3 on the same claims (strict scoring)")
    print("saved", save_figure(fig, f"08_paired_quadrants_{RID}", run)); plt.show()

## 4. A3 initial vs final verdict

Transitions against the reference, and which way the label moved.

In [ ]:
if LABELLED:
    sl = pd.DataFrame(ea.second_look_rows(preds, gold, kind_of[B]))
    display(sl.transition.value_counts().rename("claims").to_frame())
    changed = sl[sl["shift"] != "unchanged"]
    print(f"changed {len(changed)}/{len(sl)}; reconciliation did not complete (NO_FINAL_VERDICT) on "
          f"{int((~sl.completed).sum())}; guard reverted {int(sl.guard_reverted.notna().sum())}")
    display(pd.crosstab(changed.path, changed.transition, margins=True))
    display(pd.crosstab([changed["shift"], changed.gold], changed.final))
    display(pd.crosstab(sl.category, sl.transition, margins=True))
    save_table(sl, "ea_a3_second_look", run)
    paths = []
    for path, d in changed.groupby("path"):
        paths.append({"name": f"{path}  (n={len(d)})", "helped": int((d.transition == "incorrect_to_correct").sum()),
                      "harmed": int((d.transition == "correct_to_incorrect").sum())})
    paths.sort(key=lambda r: r["helped"] - r["harmed"])
    fig = plots.change_balance(paths, "A3 verdict changes by direction (I = INSUFFICIENT, S = SUPPORTED, C = CONTRADICTED)",
                               "Claims (changes that stayed wrong are not shown)")
    print("saved", save_figure(fig, f"08_change_direction_{RID}", run)); plt.show()

## 5. Errors by type, approach and category

*Overconfident*: a decisive label where the reference is INSUFFICIENT_EVIDENCE. *Over-abstention*: INSUFFICIENT_EVIDENCE where the reference is decisive. *Polarity*: SUPPORTED ↔ CONTRADICTED.

In [ ]:
if LABELLED:
    er = pd.DataFrame(ea.error_rows(preds, gold))
    et = pd.crosstab(er.approach, er.error_type).reindex(columns=list(ea.ERROR_TYPES), fill_value=0)
    display(et)
    display(pd.crosstab([er.category, er.approach], er.error_type).reindex(columns=list(ea.ERROR_TYPES), fill_value=0))
    save_table(et.reset_index(), "ea_error_types", run)
    acc_cat = er.assign(ok=er.error_type == "correct").pivot_table(index="category", columns="approach", values="ok", aggfunc="mean")
    display(acc_cat.reindex(list(plots.CATEGORIES)).round(3))
    fig = plots.error_types([{"approach": a, "kind": kind_of[a], **et.loc[a].to_dict()} for a in et.index],
                            "Final labels against the reference, by error type")
    print("saved", save_figure(fig, f"08_error_types_{RID}", run)); plt.show()
    cms = []
    for a in sorted(kind_of):
        ps = [p for p in preds if p["approach"] == a and p["claim_id"] in gold]
        cms.append((a, kind_of[a], classification_metrics([gold[p["claim_id"]]["label"] for p in ps],
                                                          [p["predicted_label"] for p in ps])["confusion_matrix"]))
    fig = plots.confusion_grid(cms, "Confusion matrices (rows: reference, columns: prediction)")
    print("saved", save_figure(fig, f"08_confusion_{RID}", run)); plt.show()

## 6. Contradiction investigation: was genuine counter-evidence retrieved and read correctly?

Automatic part (every A3 claim): what the investigation retrieved, how many challenges it judged supported with a valid citation, and whether changed verdicts rest on the counter-evidence. Manual part (the 23 A2/A3 discordant claims): `datasets/error_analysis/discordant_case_review__<run>.csv`, an AI-analyst reading of each trace (genuine counter-evidence? correctly interpreted? do the final citations entail the verdict?) awaiting researcher confirmation.

In [ ]:
aud = pd.DataFrame([ea.counter_evidence_audit(p, traces.get((p["approach"], p["claim_id"])))
                    for p in preds if p["approach"] == B])
if not aud.empty:
    assess = pd.Series([x.get("assessment") for p in preds if p["approach"] == B
                        for x in ((p.get("contradiction") or {}).get("assessments") or []) if isinstance(x, dict)]).value_counts()
    display(assess.rename("hypotheses").to_frame())
    s = {"claims": len(aud), "completed": int(aud.completed.sum()), "mean_counter_passages": aud.counter_passages.mean(),
         "claims_with_new_filings": int((aud.new_filings > 0).sum()),
         "claims_with_supported_challenge": int((aud.challenges_supported > 0).sum()),
         "supported_challenges_with_valid_citation": f"{int(aud.supported_with_valid_citation.sum())}/{int(aud.challenges_supported.sum())}",
         "changed": int(aud.changed.sum()), "changed_citing_counter_evidence": int((aud.changed & aud.final_cites_counter_evidence).sum()),
         "changed_without_supported_challenge": int(aud.changed_without_supported_challenge.sum()),
         "guard_reverted": int(aud.guard_reverted.notna().sum())}
    display(pd.Series(s, name="A3").to_frame())
    save_table(aud, "ea_counter_evidence_audit", run)
REVIEW = os.path.join("datasets", "error_analysis", f"discordant_case_review__{RID}.csv")
review = pd.read_csv(REVIEW) if os.path.exists(REVIEW) and LABELLED else None
if review is None:
    print("no discordant-case review for this run")
else:
    banner(f"Mechanisms below are an AI-analyst reading of the traces ({review.reviewer.iloc[0]}); "
           f"researcher-confirmed: {int(review.researcher_confirmed.notna().sum())}/{len(review)}.", "info")
    rv = review.merge(pair[["claim_id", "category", "gold", "label_a", "label_b", "quadrant"]], on="claim_id", how="left")
    rv["outcome"] = rv.quadrant.map({"only_a": "harmed", "only_b": "helped"})
    display(pd.crosstab(rv.mechanism, rv.outcome, margins=True))
    display(pd.crosstab(rv.outcome, [rv.counterevidence_genuine]).rename_axis(columns="genuine counter-evidence?"))
    display(pd.crosstab(rv.outcome, [rv.interpretation_correct]).rename_axis(columns="interpreted correctly?"))
    display(pd.crosstab(rv.outcome, [rv.final_citations_entail_verdict]).rename_axis(columns="final citations entail verdict?"))
    mech = [{"name": m.replace("_", " ").lower(), "helped": int((d.outcome == "helped").sum()), "harmed": int((d.outcome == "harmed").sum())}
            for m, d in rv.groupby("mechanism")]
    mech.sort(key=lambda r: (r["helped"] - r["harmed"], r["helped"]))
    fig = plots.change_balance(mech, "Why A3 changed A2's verdict: 23 discordant claims (AI-analyst coding)",
                               "Claims where A2 and A3 disagree")
    print("saved", save_figure(fig, f"08_discordant_mechanisms_{RID}", run)); plt.show()
    save_table(rv, "ea_discordant_review", run)

## 7. Citations: provenance, cutoff, period, and what a valid citation does not show

Re-checked from the trace's evidence ledger, independently of the pipeline's `valid` flag. The pipeline checks passage quotes verbatim but **not** the quotes attached to XBRL facts or derived metrics, so those are checked here by matching the cited figure against the quoted numbers. Period alignment is measured on claims that state a period end. None of these checks shows that the evidence *entails* the verdict; entailment of the discordant cases is in section 6 (manual).

In [ ]:
cit = pd.DataFrame([r for p in preds for r in ea.citation_audit(p, traces.get((p["approach"], p["claim_id"])))])
if not cit.empty:
    cs = pd.DataFrame(ea.citation_summary(cit.to_dict("records")))
    display(cs.round(3))
    save_table(cs, "ea_citation_audit_summary", run)
    save_table(cit, "ea_citation_audit", run)
    print("future-data leakage (cited filing after the claim cutoff):", int(cit.after_cutoff.sum()))
    mism = cit[cit.xbrl_value_in_quote == False]  # noqa: E712
    print("XBRL / derived-metric citations whose quote does not contain the cited figure:")
    display(mism[["approach", "claim_id", "evidence_id", "kind", "period_end"]])
    print("citations whose period differs from the claim's stated period end:")
    display(cit[cit.period_matches_claim == False].groupby(["approach", "claim_id"]).size().rename("citations").to_frame())  # noqa: E712
    decisive_no_valid = [p for p in preds if p["predicted_label"] in ea.DECISIVE and not any(c.get("valid") for c in p.get("citations") or [])]
    print("decisive answers with no valid citation:", pd.Series([p["approach"] for p in decisive_no_valid]).value_counts().to_dict())

## 8. Tool calls, tokens, cost and latency

Calls, tokens and cost come from the frozen run (identical to the live run's, which it replays). Latency comes from the live run: an offline replay records 0 ms. A request shared by approaches (A1, A2 and A3 send a byte-identical initial request) is charged to each, as if run alone.

In [ ]:
live = ea.live_latency_index([run["run_dir"]] + [os.path.join(RUNS_DIR, r) for r in SOURCE_RUNS])
eff = pd.DataFrame(ea.efficiency_rows(preds, traces, live))
lab = {a: [p for p in preds if p["approach"] == a and p["claim_id"] in gold] for a in kind_of}
summ = eff.groupby("approach").agg(claims=("claim_id", "count"), llm_calls=("llm_calls", "mean"), tool_calls=("tool_calls", "mean"),
                                   retrieval_queries=("retrieval_queries", "mean"), numeric_checks=("numeric_checks", "mean"),
                                   input_tokens=("input_tokens", "mean"), output_tokens=("output_tokens", "mean"),
                                   cost_per_claim=("cost_usd", "mean"), total_cost=("cost_usd", "sum"))
lat = eff[eff.latency_complete].groupby("approach").latency_s
summ["latency_mean_s"], summ["latency_median_s"], summ["latency_n"] = lat.mean(), lat.median(), lat.count()
if LABELLED:
    summ["correct"] = [sum(p["predicted_label"] == gold[p["claim_id"]]["label"] for p in lab[a]) for a in summ.index]
    summ["cost_per_correct"] = summ.total_cost / summ.correct
display(summ.round(3).T)
save_table(summ.reset_index(), "ea_efficiency", run)
display(eff.groupby(["approach", "category"])[["llm_calls", "tool_calls", "input_tokens", "cost_usd", "latency_s"]].mean().round(3))
if LABELLED and summ.latency_n.notna().all():
    from engine.stats import bootstrap_ci
    rows = []
    for a in summ.index:
        ci = bootstrap_ci([gold[p["claim_id"]]["label"] for p in lab[a]], [p["predicted_label"] for p in lab[a]])["accuracy"]
        rows.append({"approach": a, "kind": kind_of[a], "accuracy": ci["point"], "acc_lo": ci["ci_low"], "acc_hi": ci["ci_high"],
                     "cost_per_claim": summ.loc[a, "cost_per_claim"], "latency_s": summ.loc[a, "latency_mean_s"]})
    fig = plots.accuracy_vs_cost(rows, "Accuracy against cost and latency per claim")
    print("saved", save_figure(fig, f"08_accuracy_vs_cost_{RID}", run)); plt.show()

## 9. Failure case studies

Discordant claims from the review, with the reference rationale and the change A3 made. Full traces: `results/runs/<run>/traces/A3_contradiction_aware/<claim>.json`.

In [ ]:
if review is not None:
    idx = {(p["approach"], p["claim_id"]): p for p in preds}
    cases = []
    for _, r in rv.sort_values(["outcome", "mechanism"]).iterrows():
        p = idx[(B, r.claim_id)]
        cases.append({"claim_id": r.claim_id, "category": r.category, "outcome": r.outcome, "mechanism": r.mechanism,
                      "claim": gold[r.claim_id]["claim"], "reference": gold[r.claim_id]["label"],
                      "a2_final": r.label_a, "a3_final": r.label_b,
                      "reference_rationale": gold[r.claim_id].get("rationale"),
                      "a3_change_reason": (p.get("contradiction") or {}).get("change_reason"), "analyst_note": r.note})
    cases = pd.DataFrame(cases)
    save_table(cases, "ea_case_studies", run)
    for _, c in cases[cases.claim_id.isin(["GOLD-NFLX-09", "GOLD-NVDA-07", "GOLD-TSLA-13", "GOLD-NFLX-11", "GOLD-KO-09", "GOLD-GOOGL-04"])].iterrows():
        print("=" * 100)
        print(f"{c.claim_id} [{c.category}] {c.outcome.upper()} -- {c.mechanism}\nclaim: {c.claim}\n"
              f"reference {c.reference} | A2 {c.a2_final} -> A3 {c.a3_final}\nA3 change reason: {(c.a3_change_reason or '')[:500]}\n"
              f"analyst: {c.analyst_note}")